In [1]:
import os
import glob
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 5
IMG_SIZE = 192
BATCH_SIZE = 8
EPOCHS = 5
LR = 1e-4
HIDDEN_CH = 96

TRAIN_DIR = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/training_videos"

In [3]:
def unflip_if_needed(img):

    h = img.shape[0]
    top_mean = img[:h // 2].mean()
    bottom_mean = img[h // 2:].mean()

    if top_mean > bottom_mean:
        return np.flipud(img).copy()  
    return img.copy()

In [4]:
def preprocess_frame(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.bilateralFilter(img, 5, 75, 75)
    img = img.astype(np.float32) / 255.0
    return img

In [5]:
class AvenueDataset(Dataset):
    def __init__(self, root):
        self.samples = []

        for vid in sorted(os.listdir(root)):
            paths = sorted(glob.glob(os.path.join(root, vid, "*.jpg")))

            frames = []
            for p in paths:
                img = preprocess_frame(p)
                img = unflip_if_needed(img)
                frames.append(img)

            frames = np.stack(frames)

            for i in range(SEQ_LEN, len(frames)):
                self.samples.append((frames, i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frames, i = self.samples[idx]
        x = frames[i-SEQ_LEN:i][:, None]   # (T, 1, H, W)
        y = frames[i][None]                # (1, H, W)
        return torch.tensor(x), torch.tensor(y)

In [6]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch):
        super().__init__()
        self.hidden_ch = hidden_ch

        self.conv = nn.Conv2d(
            in_ch + hidden_ch,
            4 * hidden_ch,
            kernel_size=3,
            padding=1
        )

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)

        i, f, o, g = torch.chunk(gates, 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

In [7]:
class CNNConvLSTMPredictor(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        self.convlstm = ConvLSTMCell(64, HIDDEN_CH)

        self.decoder = nn.Sequential(
            nn.Conv2d(HIDDEN_CH, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 1, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        h = torch.zeros(B, HIDDEN_CH, H, W, device=x.device)
        c = torch.zeros_like(h)

        for t in range(T):
            feat = self.encoder(x[:, t])
            h, c = self.convlstm(feat, h, c)

        return self.decoder(h)

In [8]:
def train():
    dataset = AvenueDataset(TRAIN_DIR)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=8,
        pin_memory=True
    )

    
    model = CNNConvLSTMPredictor().to(DEVICE)
    model  = nn.DataParallel(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for x, y in tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            x, y = x.to(DEVICE), y.to(DEVICE)

            pred = model(x)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.6f}")

        torch.save(model.state_dict(), f"cnn_convlstm_epoch{epoch+1}.pth")

In [9]:
train()

KeyboardInterrupt: 

In [13]:
def extract_frame_number(path):
    return int(os.path.basename(path).split("_")[1].split(".")[0])

In [16]:
def generate_submission():
    # Load template (already has ALL required IDs)
    sample = pd.read_csv("/kaggle/input/sample/submission-8.csv")
    sample["Predicted"] = 0.0

    model = CNNConvLSTMPredictor().to(DEVICE)
    
    state = torch.load("/kaggle/input/epoch3/pytorch/default/1/cnn_convlstm_epoch3.pth", map_location=DEVICE)
    
    if any(k.startswith("module.") for k in state.keys()):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    
    model.load_state_dict(state)
    model.eval()

    score_map = {}
    all_scores = []

    for vid in tqdm(sorted(os.listdir("/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/testing_videos")), desc="Vids"):
        paths = sorted(glob.glob(os.path.join("/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset/testing_videos", vid, "*.jpg")))

        frames = [unflip_if_needed(preprocess_frame(p)) for p in paths]
        frames = np.stack(frames)                      # (N, H, W)
        frame_nums = [extract_frame_number(p) for p in paths]

        video_fids = []
        video_scores = []

        for i in range(SEQ_LEN, len(frames) - 1):
            # -------- sequence only (NO frame diff) --------
            seq = frames[i-SEQ_LEN:i]                 # (T, H, W)
            x = torch.tensor(seq, device=DEVICE) \
                    .unsqueeze(0) \
                    .unsqueeze(2)                     # (1, T, 1, H, W)

            with torch.no_grad():
                pred = model(x)[0, 0]                 # (H, W)
                gt = torch.tensor(frames[i], device=DEVICE)
                err = torch.mean((pred - gt) ** 2).item()

            video_id = str(int(vid))
            fid = f"{video_id}_{frame_nums[i]}"

            video_fids.append(fid)
            video_scores.append(err)


        video_scores = pd.Series(video_scores).ewm(alpha=0.7).mean().values

  
        video_scores[:SEQ_LEN] = 0.0

        for fid, score in zip(video_fids, video_scores):
            score_map[fid] = score
            all_scores.append(score)


    mn, mx = min(all_scores), max(all_scores)
    for k in score_map:
        score_map[k] = (score_map[k] - mn) / (mx - mn + 1e-8)

  
    for i in range(len(sample)):
        fid = sample.at[i, "Id"]
        if fid in score_map:
            sample.at[i, "Predicted"] = score_map[fid]

    sample.to_csv("cnn_convlstm_ema.csv", index=False)
    print("cnn_convlstm_ema.csv generated")

In [17]:
 generate_submission()

Vids: 100%|██████████| 21/21 [11:20<00:00, 32.42s/it]


cnn_convlstm_ema.csv generated
